# Batch Ingest — DOAC Channel

Processes multiple YouTube videos through the diarization pipeline and uploads to Supabase.

**Config:** NeMo MSDD, no Demucs, no_speech_threshold=0.1, large-v3

**Requirements:** Colab Pro (A100), HF token not needed (NeMo only)

In [ ]:
# Version: v10 (2026-04-08)
print('Batch Ingest v10 — 2026-04-08 (fix position field in segment insert)')


## 1. Setup

In [ ]:
# Clone repo
%cd /content
!rm -rf /content/vault
!git clone --branch feat/people-agents https://github.com/cha7ura/vault.git /content/vault
%cd /content/vault/vendor/whisper-diarization

In [ ]:
# Install dependencies (runtime will restart after)
!apt-get install -y -qq nodejs > /dev/null 2>&1
!pip install -q "numpy<2"
!pip install -q "faster-whisper>=1.1.0"
!pip install -q "nemo-toolkit[asr]>=2.5.0"
!pip install -q git+https://github.com/oliverguhr/deepmultilingualpunctuation.git
!pip install -q yt-dlp nltk

# Restart runtime to pick up numpy<2
import os
os._exit(0)

**After restart, run from here (cell 6) onwards. Skip cells 3-4.**

In [ ]:
import torch
import psutil

# ── Hardware detection ────────────────────────────────────────────────────────
if torch.cuda.is_available():
    _gpu_name = torch.cuda.get_device_name(0)
    _vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
else:
    _gpu_name = 'CPU (no CUDA)'
    _vram_gb  = 0.0

_ram_gb = psutil.virtual_memory().total / 1e9

# ── VRAM → batch size only (model stays large-v3) ───────────────────────────
# Only scale UP from the A100-40G baseline of 16; never reduce below it.
# If you're on a smaller GPU and hit OOM, override BATCH_SIZE manually below.
if _vram_gb >= 70:      AUTO_BATCH_SIZE = 64   # H100 80 GB
elif _vram_gb >= 38:    AUTO_BATCH_SIZE = 32   # A100 40 GB  ← current baseline
else:                   AUTO_BATCH_SIZE = 32   # keep baseline on anything lower

# ── RAM → scales from pressure-relief (low) to performance (high) ─────────────
# Thresholds are cumulative — each tier adds a capability, no upper cap needed.
AUTO_DELETE_WAV_AFTER_DECODE = _ram_gb < 30   # keep WAV on disk if room allows
AUTO_GC_BETWEEN_STAGES       = _ram_gb < 20   # skip forced GC when comfortable

# Punctuation model chunk size — larger chunks = faster batching, more RAM used
if _ram_gb >= 60:      AUTO_PUNCT_CHUNK_SIZE = 512
elif _ram_gb >= 30:    AUTO_PUNCT_CHUNK_SIZE = 350
elif _ram_gb >= 20:    AUTO_PUNCT_CHUNK_SIZE = 230
elif _ram_gb >= 10:    AUTO_PUNCT_CHUNK_SIZE = 150
else:                  AUTO_PUNCT_CHUNK_SIZE = 80

print(f'GPU : {_gpu_name}  ({_vram_gb:.1f} GB VRAM)')
print(f'RAM : {_ram_gb:.1f} GB system memory')
print()
print(f'  whisper model        : large-v3 (fixed)')
print(f'  whisper batch size   : {AUTO_BATCH_SIZE}')
print(f'  delete WAV on decode : {AUTO_DELETE_WAV_AFTER_DECODE}')
print(f'  gc between stages    : {AUTO_GC_BETWEEN_STAGES}')
print(f'  punct chunk size     : {AUTO_PUNCT_CHUNK_SIZE}')
print()
print('Override any of these in the Configuration cell below.')


## 2. Configuration

In [ ]:
from google.colab import userdata

SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = userdata.get("SUPABASE_SERVICE_ROLE_KEY")

# ============================================================
# PIPELINE SETTINGS  (auto-scaled from cell 6 — override here)
# ============================================================
WHISPER_MODEL           = "large-v3"           # fixed — do not auto-select
BATCH_SIZE              = AUTO_BATCH_SIZE       # Whisper batched-inference chunk count
DELETE_WAV_AFTER_DECODE = AUTO_DELETE_WAV_AFTER_DECODE  # free disk after decode
GC_BETWEEN_STAGES       = AUTO_GC_BETWEEN_STAGES        # gc between pipeline stages
PUNCT_CHUNK_SIZE        = AUTO_PUNCT_CHUNK_SIZE         # punct model batch chunk
DEVICE = "cuda"
NO_SPEECH_THRESHOLD = 0.1
DIARIZER = "msdd"
DIARIZER_LABEL = "nemo-msdd"

# Channel
CHANNEL_SLUG = "the-diary-of-a-ceo"

# ============================================================
# FETCH VIDEO IDS FROM SUPABASE (sorted by duration, short → long)
# ============================================================
import requests

headers = {
    "apikey": SUPABASE_SERVICE_ROLE_KEY,
    "Authorization": f"Bearer {SUPABASE_SERVICE_ROLE_KEY}",
    "Content-Type": "application/json",
    "Prefer": "return=representation",
}

def paginated_get(url, headers, page_size=1000):
    """Fetch all rows from a Supabase REST endpoint with pagination."""
    all_rows = []
    offset = 0
    while True:
        res = requests.get(
            url,
            headers={**headers, "Range": f"{offset}-{offset+page_size-1}"},
        )
        batch = res.json()
        if not isinstance(batch, list) or not batch:
            break
        all_rows.extend(batch)
        if len(batch) < page_size:
            break
        offset += page_size
    return all_rows

# Get channel ID
res = requests.get(
    f"{SUPABASE_URL}/rest/v1/channels?slug=eq.{CHANNEL_SLUG}&select=id",
    headers=headers,
)
channels = res.json()
if not channels:
    raise ValueError(f"Channel '{CHANNEL_SLUG}' not found in Supabase")
channel_id = channels[0]["id"]

# Get ALL episodes for this channel, sorted by duration (short → long)
all_episodes = paginated_get(
    f"{SUPABASE_URL}/rest/v1/episodes?channel_id=eq.{channel_id}"
    f"&select=id,youtube_id,title,duration_seconds"
    f"&order=duration_seconds.asc.nullsfirst",
    headers,
)

# Get distinct episode IDs that already have segments for this diarizer
# Supabase REST doesn't support SELECT DISTINCT, so we paginate and dedupe in Python
done_rows = paginated_get(
    f"{SUPABASE_URL}/rest/v1/segments?diarizer=eq.{DIARIZER_LABEL}"
    f"&select=episode_id",
    headers,
)
done_episode_ids = {r["episode_id"] for r in done_rows}

# Filter to episodes that haven't been processed yet
pending = [e for e in all_episodes if e["id"] not in done_episode_ids]
already_done = [e for e in all_episodes if e["id"] in done_episode_ids]

VIDEO_IDS = [e["youtube_id"] for e in pending]
episode_map = {e["youtube_id"]: e for e in all_episodes}

print(f"Channel: {CHANNEL_SLUG} ({channel_id})")
print(f"Total episodes: {len(all_episodes)}")
print(f"Already processed: {len(already_done)} ({len(done_rows)} segment rows scanned)")
print(f"Pending: {len(pending)} (sorted short → long)")
print()
for e in pending[:10]:
    dur = e.get('duration_seconds') or 0
    print(f"  {e['youtube_id']}  {dur//60:>3}m  {e.get('title', '')[:50]}")
if len(pending) > 10:
    print(f"  ... and {len(pending)-10} more")

# ============================================================
# YT CAPTION HELPERS
# ============================================================

def fetch_yt_captions(video_id):
    """Fetch YouTube auto-generated captions as word-level data."""
    import subprocess, urllib.request

    result = subprocess.run(
        ["yt-dlp", "--dump-json", "--skip-download", "--no-warnings",
         f"https://www.youtube.com/watch?v={video_id}"],
        capture_output=True, text=True, timeout=60,
    )
    if result.returncode != 0:
        return None

    meta = json.loads(result.stdout)
    en_subs = meta.get("automatic_captions", {}).get("en", [])
    json3_url = next((s["url"] for s in en_subs if s.get("ext") == "json3"), None)
    if not json3_url:
        return None

    try:
        with urllib.request.urlopen(json3_url, timeout=30) as resp:
            data = json.loads(resp.read())
    except Exception:
        return None

    words = []
    for event in data.get("events", []):
        t_start_ms = event.get("tStartMs", 0)
        for seg in (event.get("segs") or []):
            utf8 = seg.get("utf8", "").strip()
            if not utf8 or utf8 == "\n":
                continue
            word_start = (t_start_ms + seg.get("tOffsetMs", 0)) / 1000.0
            words.append({"text": utf8, "start": round(word_start, 3), "end": None})

    for i in range(len(words) - 1):
        words[i]["end"] = words[i + 1]["start"]
    if words:
        words[-1]["end"] = round(words[-1]["start"] + 0.5, 3)
    return words


def upload_yt_segments(episode_id, yt_words, segments):
    """Slice YT words into segment time boundaries and upload to yt_segments."""
    # Delete existing
    requests.delete(
        f"{SUPABASE_URL}/rest/v1/yt_segments?episode_id=eq.{episode_id}",
        headers=headers,
    )

    word_idx = 0
    rows = []
    for pos, seg in enumerate(segments):
        seg_start = seg["start"]
        seg_end = seg["end"]
        seg_words = []
        while word_idx < len(yt_words):
            w = yt_words[word_idx]
            word_mid = (w["start"] + (w["end"] or w["start"] + 0.5)) / 2
            if word_mid < seg_start:
                word_idx += 1
            elif word_mid > seg_end:
                break
            else:
                seg_words.append(w)
                word_idx += 1
        rows.append({
            "episode_id": episode_id,
            "position": pos,
            "start_time": seg_start,
            "end_time": seg_end,
            "text": " ".join(w["text"] for w in seg_words),
            "words": json.dumps(seg_words),
        })

    for b in range(0, len(rows), 500):
        batch = rows[b:b+500]
        res = requests.post(
            f"{SUPABASE_URL}/rest/v1/yt_segments",
            headers={**headers, "Prefer": "return=minimal"},
            json=batch,
        )
        if res.status_code >= 300:
            print(f"  ERROR uploading yt_segments (batch {b//500 + 1}): {res.status_code} {res.text[:200]}")
            return 0
    non_empty = sum(1 for r in rows if r["text"].strip())
    print(f"  YT captions: {len(rows)} segments ({non_empty} non-empty)")
    return len(rows)


def episode_has_yt_segments(episode_id):
    """Check if yt_segments exist for this episode."""
    res = requests.get(
        f"{SUPABASE_URL}/rest/v1/yt_segments?episode_id=eq.{episode_id}&select=id&limit=1",
        headers={**headers, "Prefer": "count=exact"},
    )
    count = res.headers.get("content-range", "*/0").split("/")[-1]
    return int(count) > 0


# ============================================================
# POPULATE NEW EPISODES FROM CHANNEL
# ============================================================

def populate_new_episodes():
    """Discover new videos on the channel and add to episodes table."""
    import subprocess
    result = subprocess.run(
        ["yt-dlp", "--flat-playlist", "--dump-json",
         f"https://www.youtube.com/@TheDiaryOfACEO/videos"],
        capture_output=True, text=True, timeout=120,
    )
    if result.returncode != 0:
        print(f"  yt-dlp error: {result.stderr[:300]}")
        return 0

    existing = {e["youtube_id"] for e in all_episodes}
    new_count = 0
    for line in result.stdout.strip().split("\n"):
        if not line.strip():
            continue
        try:
            meta = json.loads(line)
        except json.JSONDecodeError:
            continue
        vid = meta.get("id")
        if not vid or vid in existing:
            continue
        title = meta.get("title", f"Episode {vid}")
        duration = meta.get("duration")
        res = requests.post(
            f"{SUPABASE_URL}/rest/v1/episodes",
            headers=headers,
            json={
                "youtube_id": vid,
                "channel_id": channel_id,
                "title": title,
                "duration_seconds": int(duration) if duration else None,
            },
        )
        if res.status_code < 300:
            new_count += 1
            existing.add(vid)
    return new_count


# ============================================================
# BACKFILL YT CAPTIONS FOR EXISTING EPISODES
# ============================================================

def backfill_yt_captions():
    """Fetch YT captions for episodes that have segments but no yt_segments."""
    import time as _time
    import random as _random

    # Get episodes with segments
    done_rows = paginated_get(
        f"{SUPABASE_URL}/rest/v1/segments?diarizer=eq.{DIARIZER_LABEL}&select=episode_id",
        headers,
    )
    episodes_with_segs = {r["episode_id"] for r in done_rows}

    to_backfill = []
    for ep in all_episodes:
        if ep["id"] in episodes_with_segs and not episode_has_yt_segments(ep["id"]):
            to_backfill.append(ep)

    print(f"Episodes needing YT caption backfill: {len(to_backfill)}")

    success = 0
    for i, ep in enumerate(to_backfill, 1):
        title = (ep.get("title") or ep["youtube_id"])[:50]
        print(f"  [{i}/{len(to_backfill)}] {ep['youtube_id']} — {title}")

        yt_words = fetch_yt_captions(ep["youtube_id"])
        if not yt_words:
            print(f"    No captions available")
            _time.sleep(15 + _random.random() * 15)
            continue

        # Get segment boundaries
        seg_rows = paginated_get(
            f"{SUPABASE_URL}/rest/v1/segments?episode_id=eq.{ep['id']}"
            f"&diarizer=eq.{DIARIZER_LABEL}&select=start_time,end_time&order=start_time",
            headers,
        )
        seg_list = [{"start": s["start_time"], "end": s["end_time"]} for s in seg_rows]
        upload_yt_segments(ep["id"], yt_words, seg_list)
        success += 1
        _time.sleep(15 + _random.random() * 15)

    print(f"Backfill done: {success}/{len(to_backfill)}")
    return success


## 3. Load Models Once

In [ ]:
import os, sys, time, json
import numpy as np
import torch
import faster_whisper
import requests

import nltk
nltk.download('punkt_tab', quiet=True)

WHISPER_DIR = "/content/vault/vendor/whisper-diarization"
os.chdir(WHISPER_DIR)
sys.path.insert(0, WHISPER_DIR)

from deepmultilingualpunctuation import PunctuationModel
from helpers import (
    find_numeral_symbol_tokens,
    get_realigned_ws_mapping_with_punctuation,
    get_sentences_speaker_mapping,
    get_words_speaker_mapping,
    punct_model_langs,
)
from diarization import MSDDDiarizer

mtypes = {"cpu": "int8", "cuda": "float16"}

# --- Load all models ---
print("[1/3] Loading Whisper model...")
t0 = time.time()
whisper_model = faster_whisper.WhisperModel(
    WHISPER_MODEL, device=DEVICE, compute_type=mtypes[DEVICE]
)
whisper_pipeline = faster_whisper.BatchedInferencePipeline(whisper_model)
print(f"  Whisper loaded in {time.time()-t0:.1f}s")

print("[2/3] Loading MSDD diarizer (downloads VAD + TitaNet + MSDD from NGC)...")
t0 = time.time()
diarizer_model = MSDDDiarizer(device=DEVICE)
print(f"  MSDD loaded in {time.time()-t0:.1f}s")

# Warmup: run a 1-second silent audio through MSDD to force all sub-model downloads
print("  Warming up MSDD (downloading sub-models if needed)...")
warmup_audio = torch.zeros(1, 16000)  # 1 second silence
try:
    _ = diarizer_model.diarize(warmup_audio)
except:
    pass  # warmup may fail on silence, that's fine — models are now cached
print("  Warmup done — all NGC models cached")

print("[3/3] Loading punctuation model...")
t0 = time.time()
punct_model = PunctuationModel(model="oliverguhr/fullstop-punctuation-multilang-large")
print(f"  Punctuation loaded in {time.time()-t0:.1f}s")

print(f"\nAll models loaded and cached. GPU peak: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

## 4. Populate New Episodes + Backfill YT Captions

Discovers new videos on the channel, adds them to the episodes table, then backfills YT captions for episodes that have segments but no yt_segments.

In [ ]:
# Step 4a: Add any new videos to episodes table
print('Populating new episodes from channel...')
new_count = populate_new_episodes()
print(f'  Added {new_count} new episodes')

# Refresh episode list
all_episodes = paginated_get(
    f"{SUPABASE_URL}/rest/v1/episodes?channel_id=eq.{channel_id}"
    f"&select=id,youtube_id,title,duration_seconds"
    f"&order=duration_seconds.asc.nullsfirst",
    headers,
)
print(f'  Total episodes: {len(all_episodes)}')

# Step 4b: Backfill YT captions for episodes with segments
print('\nBackfilling YT captions...')
backfill_yt_captions()

# Recalculate pending (in case new episodes were added)
done_rows = paginated_get(
    f"{SUPABASE_URL}/rest/v1/segments?diarizer=eq.{DIARIZER_LABEL}&select=episode_id",
    headers,
)
done_episode_ids = {r['episode_id'] for r in done_rows}
pending = [e for e in all_episodes if e['id'] not in done_episode_ids]
VIDEO_IDS = [e['youtube_id'] for e in pending]
print(f'\nPending for diarization: {len(VIDEO_IDS)} episodes')


## 5. Process All Videos

In [ ]:
import re

def download_audio(video_id):
    """Download YouTube audio as 16kHz mono WAV."""
    audio_path = f"/content/{video_id}.wav"
    if os.path.exists(audio_path):
        print(f"  Audio already exists: {audio_path}")
        return audio_path

    os.chdir("/content")
    ret = os.system(
        f'yt-dlp -x --audio-format wav --postprocessor-args "ffmpeg:-ar 16000 -ac 1" '
        f'-o "{video_id}.%(ext)s" "https://www.youtube.com/watch?v={video_id}"'
    )
    os.chdir(WHISPER_DIR)

    if ret != 0 or not os.path.exists(audio_path):
        print(f"  ERROR: Failed to download {video_id}")
        return None
    return audio_path


def transcribe(audio_path):
    """Run Whisper transcription using pre-loaded model."""
    audio_waveform = faster_whisper.decode_audio(audio_path)
    if DELETE_WAV_AFTER_DECODE and os.path.exists(audio_path):
        os.remove(audio_path)  # free disk early; waveform is already in RAM

    segments, info = whisper_pipeline.transcribe(
        audio_waveform, "en",
        suppress_tokens=[-1],
        batch_size=BATCH_SIZE,
        word_timestamps=True,
        no_speech_threshold=NO_SPEECH_THRESHOLD,
    )

    word_timestamps = []
    for seg in segments:
        if seg.words:
            for w in seg.words:
                word_timestamps.append({
                    "text": w.word.strip(),
                    "start": w.start,
                    "end": w.end,
                    "score": w.probability,
                })

    return audio_waveform, word_timestamps, info.language


def diarize(audio_waveform):
    """Run MSDD diarization using pre-loaded model."""
    return diarizer_model.diarize(torch.from_numpy(audio_waveform).unsqueeze(0))


def postprocess(word_timestamps, speaker_ts, language):
    """Map speakers to words, restore punctuation, build segments."""
    wsm = get_words_speaker_mapping(word_timestamps, speaker_ts, "start")

    if language in punct_model_langs:
        words_list = [w["word"] for w in wsm]
        try:
            labeled = punct_model.predict(words_list, chunk_size=PUNCT_CHUNK_SIZE)
        except TypeError:
            labeled = punct_model.predict(words_list)

        ending_puncts = ".?!"
        model_puncts = ".,;:!?"
        is_acronym = lambda x: re.fullmatch(r"\b(?:[a-zA-Z]\.){2,}", x)

        for wd, labeled_tuple in zip(wsm, labeled):
            word = wd["word"]
            if word and labeled_tuple[1] in ending_puncts and (word[-1] not in model_puncts or is_acronym(word)):
                word += labeled_tuple[1]
                if word.endswith(".."):
                    word = word.rstrip(".")
                wd["word"] = word

    wsm = get_realigned_ws_mapping_with_punctuation(wsm)
    ssm = get_sentences_speaker_mapping(wsm, speaker_ts)

    # Build JSON segments with per-word data
    json_segments = []
    wsm_idx = 0
    for sentence in ssm:
        seg_words = []
        s_start = sentence["start_time"]
        s_end = sentence["end_time"]
        while wsm_idx < len(wsm):
            w = wsm[wsm_idx]
            if w["start_time"] >= s_start and w["start_time"] <= s_end:
                seg_words.append({
                    "text": w["word"],
                    "start": round(w["start_time"] / 1000, 3),
                    "end": round(w["end_time"] / 1000, 3),
                    "score": w.get("score"),
                })
                wsm_idx += 1
            elif w["start_time"] > s_end:
                break
            else:
                wsm_idx += 1

        json_segments.append({
            "speaker": sentence["speaker"],
            "start": round(s_start / 1000, 3),
            "end": round(s_end / 1000, 3),
            "text": sentence["text"].strip(),
            "words": seg_words,
        })

    return json_segments


def upload_to_supabase(video_id, segments, duration_s):
    """Create/update episode and upload segments."""
    # Get or create episode
    res = requests.get(
        f"{SUPABASE_URL}/rest/v1/episodes?youtube_id=eq.{video_id}&channel_id=eq.{channel_id}&select=id",
        headers=headers,
    )
    episodes = res.json()

    if episodes:
        episode_id = episodes[0]["id"]
        # Delete existing segments for this diarizer
        requests.delete(
            f"{SUPABASE_URL}/rest/v1/segments?episode_id=eq.{episode_id}&diarizer=eq.{DIARIZER_LABEL}",
            headers=headers,
        )
    else:
        res = requests.post(
            f"{SUPABASE_URL}/rest/v1/episodes",
            headers=headers,
            json={
                "youtube_id": video_id,
                "channel_id": channel_id,
                "title": f"Episode {video_id}",
                "duration_seconds": int(duration_s),
            },
        )
        if res.status_code >= 300:
            print(f"  ERROR creating episode: {res.text}")
            return None, 0
        episode_id = res.json()[0]["id"]

    # Upload segments in batches
    rows = [{
        "episode_id": episode_id,
        "position": i,
        "start_time": seg["start"],
        "end_time": seg["end"],
        "text": seg["text"],
        "speaker": seg["speaker"],
        "words": json.dumps(seg.get("words", [])),
        "diarizer": DIARIZER_LABEL,
    } for i, seg in enumerate(segments)]

    for b in range(0, len(rows), 100):
        batch = rows[b:b+100]
        res = requests.post(
            f"{SUPABASE_URL}/rest/v1/segments",
            headers={**headers, "Prefer": "return=minimal"},
            json=batch,
        )
        if res.status_code >= 300:
            print(f"  ERROR uploading segments (batch {b//100 + 1}, rows {b}-{b+len(batch)-1}): {res.status_code} {res.text[:200]}")
            return None, 0

    return episode_id, len(rows)


print("Functions ready")

In [ ]:
# ============================================================
# PROCESS ALL VIDEOS
# ============================================================
import gc

results = []
total_start = time.time()

for idx, video_id in enumerate(VIDEO_IDS):
    print(f"\n{'='*70}")
    print(f"[{idx+1}/{len(VIDEO_IDS)}] Processing: {video_id}")
    print(f"{'='*70}")

    try:
        # 1. Download
        t0 = time.time()
        audio_path = download_audio(video_id)
        if not audio_path:
            results.append({"video_id": video_id, "status": "download_failed"})
            continue
        dl_time = time.time() - t0

        # 2. Transcribe
        print(f"  Transcribing...")
        t0 = time.time()
        waveform, word_timestamps, language = transcribe(audio_path)
        whisper_time = time.time() - t0
        duration_s = len(waveform) / 16000
        print(f"  Whisper: {len(word_timestamps)} words in {whisper_time:.1f}s ({duration_s:.0f}s audio)")

        # 3. Diarize
        print(f"  Diarizing (MSDD)...")
        t0 = time.time()
        speaker_ts = diarize(waveform)
        diarize_time = time.time() - t0
        print(f"  MSDD: {len(speaker_ts)} turns in {diarize_time:.1f}s")

        # Free waveform — not needed beyond diarization
        del waveform
        if GC_BETWEEN_STAGES:
            gc.collect()
            torch.cuda.empty_cache()

        # 4. Post-process
        print(f"  Post-processing...")
        t0 = time.time()
        segments = postprocess(word_timestamps, speaker_ts, language)
        post_time = time.time() - t0
        speakers = {s["speaker"] for s in segments}
        print(f"  Output: {len(segments)} segments, {len(speakers)} speakers in {post_time:.1f}s")

        # 5. Upload
        print(f"  Uploading to Supabase...")
        episode_id, n_uploaded = upload_to_supabase(video_id, segments, duration_s)
        if episode_id is None:
            results.append({"video_id": video_id, "status": "upload_failed"})
            del word_timestamps, speaker_ts, segments
            gc.collect(); torch.cuda.empty_cache()
            continue
        print(f"  Uploaded {n_uploaded} segments")

        # 5b. Fetch and upload YT captions
        print(f"  Fetching YT captions...")
        yt_words = fetch_yt_captions(video_id)
        if yt_words:
            upload_yt_segments(episode_id, yt_words, segments)
        else:
            print(f"  No YT captions available")

        # 6. Save JSON locally
        json_path = f"/content/{video_id}.json"
        with open(json_path, "w") as f:
            json.dump(segments, f, ensure_ascii=False, indent=2)

        total_time = dl_time + whisper_time + diarize_time + post_time
        results.append({
            "video_id": video_id,
            "status": "ok",
            "duration_s": round(duration_s),
            "words": len(word_timestamps),
            "segments": len(segments),
            "speakers": len(speakers),
            "whisper_time": round(whisper_time, 1),
            "diarize_time": round(diarize_time, 1),
            "total_time": round(total_time, 1),
        })
        # Free GPU memory from this episode
        del word_timestamps, speaker_ts, segments
        gc.collect()
        torch.cuda.empty_cache()

        print(f"  DONE in {total_time:.1f}s (GPU: {torch.cuda.memory_allocated()/1e9:.1f}GB)")

    except Exception as e:
        print(f"  ERROR: {e}")
        results.append({"video_id": video_id, "status": f"error: {e}"})

total_elapsed = time.time() - total_start
print(f"\n{'='*70}")
print(f"ALL DONE — {len(VIDEO_IDS)} videos in {total_elapsed:.0f}s")
print(f"{'='*70}")

## 6. Summary

In [ ]:
import psutil

print(f"{'Video ID':<15} {'Status':>8} {'Duration':>10} {'Words':>8} {'Segments':>10} {'Speakers':>10} {'Time':>8}")
print("-" * 80)
for r in results:
    if r["status"] == "ok":
        print(f"{r['video_id']:<15} {'OK':>8} {r['duration_s']:>9}s {r['words']:>8} {r['segments']:>10} {r['speakers']:>10} {r['total_time']:>7.1f}s")
    else:
        print(f"{r['video_id']:<15} {'FAIL':>8} {r['status']}")

ok = [r for r in results if r['status'] == 'ok']
if ok:
    total_audio = sum(r['duration_s'] for r in ok)
    total_proc = sum(r['total_time'] for r in ok)
    print(f"\nTotal audio: {total_audio/60:.0f} min")
    print(f"Total processing: {total_proc/60:.1f} min")
    print(f"Speed: {total_audio/total_proc:.1f}x realtime")

# GPU/RAM
max_mem = torch.cuda.max_memory_allocated() / 1e9
total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
ram = psutil.virtual_memory()
print(f"\nGPU: Peak {max_mem:.2f} GB / {total_mem:.1f} GB ({max_mem/total_mem*100:.1f}%)")
print(f"RAM: {ram.used/1e9:.1f} GB / {ram.total/1e9:.1f} GB ({ram.percent}%)")